**Pipeline preprocessing gambar di Python** menggunakan **Pandas + OpenCV + TensorFlow/Keras ImageDataGenerator**. Pipeline ini akan:

1. Membaca file CSV
2. Mengambil path gambar dari kolom `image_path`
3. Melakukan:

   * **Resizing** menjadi ukuran 224 x 224
   * **Normalisasi** ke dalam 0-1 
   * **Augmentasi**:
   
| Augmentasi       | Fungsi            |
| ---------------- | ----------------- |
| horizontal_flip  | membalik gambar   |
| rotation_range   | rotasi acak 45, 90, 180      |
| brightness_range | variasi kecerahan |

4. Menyimpan hasil preprocessing ke folder baru

In [3]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

2026-05-05 16:11:29.331295: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-05 16:11:29.879891: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-05 16:11:29.884125: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-05 16:11:31.832198: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [5]:
# ==========================
# Load CSV
# ==========================
df = pd.read_csv(r"/home/g6422078/Merge dgn data bang abin/df_merged_NA_BA_double_extension_fixed.csv")
#df_val = pd.read_csv(r"/home/g6422078/Data Splitting/df_val.csv")
#df_test = pd.read_csv(r"/home/g6422078/Data Splitting/df_test.csv")


# ==========================
# Folder output
# ==========================
output_dir = r"/home/g6422078/Data Preprocessing/Training Set/NA x BA"
nonaugmented_dir = r"/home/g6422078/Data Preprocessing/Training Set/NA x BA/Nonaugmented"
flip_dir = r"/home/g6422078/Data Preprocessing/Training Set/NA x BA/Horizontal Flip"
rot_dir = r"/home/g6422078/Data Preprocessing/Training Set/NA x BA/Rotated"
bright_dir = r"/home/g6422078/Data Preprocessing/Training Set/NA x BA/Brightness Varied"



os.makedirs(output_dir, exist_ok=True)
os.makedirs(nonaugmented_dir, exist_ok=True)
os.makedirs(flip_dir, exist_ok=True)
os.makedirs(rot_dir, exist_ok=True)
os.makedirs(bright_dir, exist_ok=True)
#os.makedirs(val_dir, exist_ok=True)
#os.makedirs(test_dir, exist_ok=True)


In [20]:
df.value_counts(['Cluster'])

Cluster
4          536
1          498
3          438
2          434
0          337
Name: count, dtype: int64

In [17]:
# ==========================
# Resize + Normalization
# ==========================
preprocess = keras.Sequential([
    layers.Resizing(224, 224),
    layers.Rescaling(1./255)
])

# ==========================
# Augmentation pipelines
# ==========================
flip_aug = keras.Sequential([
    layers.RandomFlip("horizontal")
])

rotation_aug = keras.Sequential([
    layers.RandomRotation(0.5)  # cukup besar agar bisa mendekati 45–180
])

brightness_aug = keras.Sequential([
    layers.RandomBrightness(factor=0.3, value_range=[0.0,1.0], seed=42) # value_range=[0.0,1.0] karena the pictures are already being scaled ke rentang [0,1] sebelumnya
])



In [40]:
# ==========================
# Loop setiap gambar - NONAUGMENTED
# ==========================
for path in df["image_path"]:

    try:
        img = Image.open(path).convert("RGB")
        img = np.array(img)

        # tambah batch dimension
        img = np.expand_dims(img, axis=0)

        # preprocessing --> pada seluruh dataset
        img = preprocess(img)
        
        # convert TensorFlow EagerTensor menjadi numpy agar bisa disimpan menjadi file gambar
        img = np.squeeze(img.numpy())


        filename = os.path.basename(path)
        
        # simpan ke direktori
        
        Image.fromarray((img * 255).astype(np.uint8)).save(
            os.path.join(nonaugmented_dir, filename)
        )

    except Exception as e:
        print(f"Gagal memproses: {path}")
        print(e)

Gagal memproses: /home/g6422078/Dataset/Far/file-20190731-186824-fdcdor.avif
[Errno 2] No such file or directory: '/home/g6422078/Dataset/Far/file-20190731-186824-fdcdor.avif'


In [19]:
# ==========================
# Loop setiap gambar - AUGMENTED
# ==========================
for path in df["image_path"]:

    try:
        img = Image.open(path).convert("RGB")
        img = np.array(img)

        # tambah batch dimension
        img = np.expand_dims(img, axis=0)

        # preprocessing 
        img = preprocess(img)

        filename = os.path.basename(path)

        # ==================
        # Horizontal Flip --> hanya pada training set 
        # ==================
        flip_img = flip_aug(img, training=True)
        flip_img = np.squeeze(flip_img.numpy())

        Image.fromarray((flip_img * 255).astype(np.uint8)).save(
            os.path.join(flip_dir, filename)
        )

        # ==================
        # Rotation --> hanya pada training set
        # ==================
        rot_img = rotation_aug(img, training=True)
        rot_img = np.squeeze(rot_img.numpy())

        Image.fromarray((rot_img * 255).astype(np.uint8)).save(
            os.path.join(rot_dir, filename)
        )

        # ==================
        # Brightness --> hanya pada training set
        # ==================
        bright_img = brightness_aug(img, training=True)
        bright_img = np.squeeze(bright_img.numpy())

        Image.fromarray((bright_img * 255).astype(np.uint8)).save(
            os.path.join(bright_dir, filename)
        )

    except Exception as e:
        print(f"Gagal memproses: {path}")
        print(e)

Gagal memproses: /home/g6422078/Dataset/Far/file-20190731-186824-fdcdor.avif
[Errno 2] No such file or directory: '/home/g6422078/Dataset/Far/file-20190731-186824-fdcdor.avif'


In [25]:


def create_augmented_df(folder, base_df):

    #files = os.listdir(folder)
    
    base_df = base_df.copy()
    base_df["filename"] = base_df["image_path"].apply(os.path.basename)
    
    filename_to_label = dict(zip(base_df["filename"], base_df["Cluster"]))

    data = []

    for f in os.listdir(folder):
        if f in filename_to_label:
            data.append({
                "image_path": os.path.join(folder, f),
                "Cluster": filename_to_label[f]
            })
            

    return pd.DataFrame(data)

nonaugmented_df = create_augmented_df(nonaugmented_dir, df) # namanya create augmented, salah itu, harusnya create preprocessed. tapi aku males ganti hehe
brightness_df = create_augmented_df(bright_dir, df)
rotated_df    = create_augmented_df(rot_dir, df)
flip_df = create_augmented_df(flip_dir, df)

In [30]:
# ambil filename dari df asli
df_filenames = set(df["image_path"].apply(os.path.basename))

# ambil filename dari hasil mapping
nonaug_filenames = set(nonaugmented_df["image_path"].apply(os.path.basename))

# cari yang hilang
missing_files = df_filenames - nonaug_filenames

print("Jumlah file hilang:", len(missing_files))
print("File yang hilang:", missing_files)

Jumlah file hilang: 1
File yang hilang: {'file-20190731-186824-fdcdor.avif'}


In [26]:
nonaugmented_df.value_counts(['Cluster'])

Cluster
4          535
1          498
3          438
2          434
0          337
Name: count, dtype: int64

In [27]:
brightness_df.value_counts(['Cluster'])

Cluster
4          535
1          498
3          438
2          434
0          337
Name: count, dtype: int64

In [28]:
rotated_df.value_counts(['Cluster'])

Cluster
4          535
1          498
3          438
2          434
0          337
Name: count, dtype: int64

In [29]:
flip_df.value_counts(['Cluster'])

Cluster
4          535
1          498
3          438
2          434
0          337
Name: count, dtype: int64

In [31]:
preprocessed_full = pd.concat([nonaugmented_df,
                        brightness_df,
                        rotated_df,
                        flip_df],
                        ignore_index=True)

preprocessed_full_with_original_data = pd.concat([df[["image_path","Cluster"]],
                        nonaugmented_df,                          
                        brightness_df,
                        rotated_df,
                        flip_df],
                        ignore_index=True)

In [32]:
preprocessed_full

,image_path,Cluster
0,/home/g6422078/Data Preprocessing/Training Set...,1
1,/home/g6422078/Data Preprocessing/Training Set...,1
2,/home/g6422078/Data Preprocessing/Training Set...,2
3,/home/g6422078/Data Preprocessing/Training Set...,4
4,/home/g6422078/Data Preprocessing/Training Set...,4
...,...,...
8963,/home/g6422078/Data Preprocessing/Training Set...,4
8964,/home/g6422078/Data Preprocessing/Training Set...,4
8965,/home/g6422078/Data Preprocessing/Training Set...,1
8966,/home/g6422078/Data Preprocessing/Training Set...,4


In [44]:
preprocessed_full_with_original_data.tail()

,image_path,Cluster
11190,/home/g6422078/Data Preprocessing/Training Set...,4
11191,/home/g6422078/Data Preprocessing/Training Set...,4
11192,/home/g6422078/Data Preprocessing/Training Set...,4
11193,/home/g6422078/Data Preprocessing/Training Set...,4
11194,/home/g6422078/Data Preprocessing/Training Set...,1


In [33]:
preprocessed_full.to_csv(r"/home/g6422078/Data Preprocessing/Training Set/NA x BA/df_preprocessed_NA_BA (revisi pada nilai cluster data).csv",index=False)
#preprocessed_full_with_original_data.to_csv(r"/home/g6422078/Data Preprocessing/Training Set/NA x BA/df_preprocessed_plus_original_NA_BA.csv",index=False)


In [47]:
preprocessed_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8952 entries, 0 to 8951
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   image_path  8952 non-null   object
 1   Cluster     8952 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 140.0+ KB


In [34]:
import torch
print(torch.backends.cudnn.enabled)

True
